# PPO Curriculum Agent for Bluebird Gymnasium

This notebook is a **follow-on** to [minimal_ppo_agent.ipynb](/home/sprite/BluebirdATC/bluebird-gymnasium/examples/minimal_ppo_agent.ipynb).

It keeps the same overall notebook style and PPO structure, but extends the training setup to support:

- staged curriculum learning
- checkpoint reuse across stages and notebook sessions
- gradual aircraft-count increases
- a larger-sector follow-on stage after Sector I

The important design choice is that the curriculum stays **decentralized** and **lateral-only**.
That keeps the observation dimension and action count stable, so checkpoints remain reusable.


## Environment note

This notebook can bootstrap `torch` into the current Python/Jupyter kernel if it
is missing. It prefers `uv pip install --python <current-kernel>` when `uv` is
available, and falls back to `python -m pip install` otherwise. Bluebird project
dependencies still need to be available in the kernel or local workspace.

It is intentionally based on the older `minimal_ppo_agent.ipynb` line rather than replacing it.


In [1]:
from __future__ import annotations

import importlib
import shutil
import subprocess
import sys


def ensure_python_package(module_name: str, install_name: str | None = None) -> None:
    if importlib.util.find_spec(module_name) is not None:
        return

    package_name = install_name or module_name
    uv_executable = shutil.which('uv')

    if uv_executable is not None:
        print(f'Installing {package_name} with uv into the current kernel environment...')
        subprocess.check_call([
            uv_executable,
            'pip',
            'install',
            '--python',
            sys.executable,
            package_name,
        ])
        return

    try:
        import pip  # noqa: F401
    except ImportError:
        import ensurepip
        ensurepip.bootstrap(upgrade=True)

    print(f'Installing {package_name} with pip into the current kernel environment...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])


ensure_python_package('torch')

import torch

print(f'Torch available in kernel: {torch.__version__}')


Torch available in kernel: 2.12.0+cu130


## Imports and path setup

This cell makes the notebook runnable from either the repo root or the
`bluebird-gymnasium` directory by adding the local package paths to `sys.path`.


In [2]:
from __future__ import annotations

import json
import random
import shutil
import sys
from pathlib import Path

import imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import Image, display

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
gym_root = None
dt_root = None

for candidate in search_roots:
    if (candidate / 'bluebird_gymnasium').exists():
        gym_root = candidate
        sibling_dt = candidate.parent / 'bluebird-dt'
        if sibling_dt.exists():
            dt_root = sibling_dt
        break
    if (candidate / 'bluebird-gymnasium').exists() and (candidate / 'bluebird-dt').exists():
        gym_root = candidate / 'bluebird-gymnasium'
        dt_root = candidate / 'bluebird-dt'
        break

if gym_root is None or dt_root is None:
    raise RuntimeError('Could not locate local bluebird-gymnasium and bluebird-dt package roots.')

examples_root = gym_root / 'examples'
sys.path.insert(0, str(gym_root))
sys.path.insert(0, str(dt_root))
sys.path.insert(0, str(examples_root))

from bluebird_gymnasium.envs import EnvConfig, ViewType
from bluebird_gymnasium.envs.sector_i import SectorIEnv
from bluebird_gymnasium.envs.sector_xplus import SectorXPlusEnv
from curriculum_scenarios import register_curriculum_scenario_managers

register_curriculum_scenario_managers()

print(f'Using bluebird-gymnasium from: {gym_root}')
print(f'Using bluebird-dt from: {dt_root}')
print(f'Using notebook helpers from: {examples_root}')
print(f'Torch version: {torch.__version__}')


Using bluebird-gymnasium from: /mnt/74F4CA39F4C9FCFC/Giles/Projects/project-bluebird/BluebirdATC/bluebird-gymnasium
Using bluebird-dt from: /mnt/74F4CA39F4C9FCFC/Giles/Projects/project-bluebird/BluebirdATC/bluebird-dt
Using notebook helpers from: /mnt/74F4CA39F4C9FCFC/Giles/Projects/project-bluebird/BluebirdATC/bluebird-gymnasium/examples
Torch version: 2.12.0+cu130


## PPO actor-critic model and agents

This is the same style of PPO agent used in the older minimal PPO notebook.
The main additions are shape-aware checkpoint loading and explicit metadata storage.


In [3]:
class ActorCriticNetwork(nn.Module):
    """Shared-trunk actor-critic network for PPO."""

    def __init__(
        self,
        observation_dimension: int,
        number_of_actions: int,
        hidden_units: int = 128,
    ) -> None:
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(observation_dimension, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
        )
        self.policy_head = nn.Linear(hidden_units, number_of_actions)
        self.value_head = nn.Linear(hidden_units, 1)

    def forward(self, observation_batch: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        features = self.trunk(observation_batch)
        action_logits = self.policy_head(features)
        state_value = self.value_head(features).squeeze(-1)
        return action_logits, state_value


class PPOAgent:
    """Minimal PPO agent with actor-critic network and checkpoint helpers."""

    def __init__(
        self,
        observation_dimension: int,
        number_of_actions: int,
        learning_rate: float = 3e-4,
        hidden_units: int = 128,
        clip_epsilon: float = 0.2,
        value_loss_coefficient: float = 0.5,
        entropy_coefficient: float = 0.01,
        ppo_epochs: int = 4,
    ) -> None:
        self.observation_dimension = observation_dimension
        self.number_of_actions = number_of_actions
        self.actor_critic = ActorCriticNetwork(
            observation_dimension=observation_dimension,
            number_of_actions=number_of_actions,
            hidden_units=hidden_units,
        )
        self.optimizer = optim.Adam(self.actor_critic.parameters(), lr=learning_rate)
        self.clip_epsilon = clip_epsilon
        self.value_loss_coefficient = value_loss_coefficient
        self.entropy_coefficient = entropy_coefficient
        self.ppo_epochs = ppo_epochs

    def choose_training_action(
        self,
        observation_vector: np.ndarray,
    ) -> tuple[int, torch.Tensor, torch.Tensor, torch.Tensor]:
        observation_tensor = torch.tensor(
            observation_vector,
            dtype=torch.float32,
        ).unsqueeze(0)
        action_logits, state_value = self.actor_critic(observation_tensor)
        action_distribution = torch.distributions.Categorical(logits=action_logits)
        sampled_action = action_distribution.sample()
        action_log_probability = action_distribution.log_prob(sampled_action)

        return (
            sampled_action.item(),
            observation_tensor.squeeze(0),
            action_log_probability.squeeze(0),
            state_value.squeeze(0),
        )

    def choose_evaluation_actions(
        self,
        observation_by_callsign: dict[str, np.ndarray],
    ) -> dict[str, int]:
        chosen_actions: dict[str, int] = {}
        with torch.no_grad():
            for callsign, observation_vector in observation_by_callsign.items():
                observation_tensor = torch.tensor(
                    observation_vector,
                    dtype=torch.float32,
                ).unsqueeze(0)
                action_logits, _state_value = self.actor_critic(observation_tensor)
                chosen_actions[callsign] = torch.argmax(action_logits, dim=-1).item()
        return chosen_actions

    def update_from_trajectory(
        self,
        observations: list[torch.Tensor],
        actions: list[torch.Tensor],
        old_log_probabilities: list[torch.Tensor],
        returns: torch.Tensor,
        advantages: torch.Tensor,
    ) -> dict[str, float] | None:
        if not observations:
            return None

        observation_tensor = torch.stack(observations)
        action_tensor = torch.stack(actions).long()
        old_log_probability_tensor = torch.stack(old_log_probabilities).detach()
        returns_tensor = returns.detach()
        advantages_tensor = advantages.detach()

        if advantages_tensor.numel() > 1:
            advantages_std = advantages_tensor.std(unbiased=False)
            if advantages_std > 1e-8:
                advantages_tensor = (
                    (advantages_tensor - advantages_tensor.mean())
                    / (advantages_std + 1e-8)
                )

        mean_policy_loss = 0.0
        mean_value_loss = 0.0
        mean_entropy = 0.0
        mean_total_loss = 0.0

        for _epoch in range(self.ppo_epochs):
            new_action_logits, new_state_values = self.actor_critic(observation_tensor)
            action_distribution = torch.distributions.Categorical(logits=new_action_logits)
            new_log_probabilities = action_distribution.log_prob(action_tensor)
            entropy = action_distribution.entropy().mean()

            probability_ratio = torch.exp(new_log_probabilities - old_log_probability_tensor)
            unclipped_objective = probability_ratio * advantages_tensor
            clipped_objective = torch.clamp(
                probability_ratio,
                1.0 - self.clip_epsilon,
                1.0 + self.clip_epsilon,
            ) * advantages_tensor

            policy_loss = -torch.min(unclipped_objective, clipped_objective).mean()
            value_loss = torch.nn.functional.mse_loss(new_state_values, returns_tensor)
            total_loss = (
                policy_loss
                + self.value_loss_coefficient * value_loss
                - self.entropy_coefficient * entropy
            )

            self.optimizer.zero_grad()
            total_loss.backward()
            self.optimizer.step()

            mean_policy_loss += float(policy_loss.item())
            mean_value_loss += float(value_loss.item())
            mean_entropy += float(entropy.item())
            mean_total_loss += float(total_loss.item())

        epoch_divisor = float(self.ppo_epochs)
        return {
            'policy_loss': mean_policy_loss / epoch_divisor,
            'value_loss': mean_value_loss / epoch_divisor,
            'entropy': mean_entropy / epoch_divisor,
            'total_loss': mean_total_loss / epoch_divisor,
        }

    def save_checkpoint(self, checkpoint_path: Path, metadata: dict | None = None) -> None:
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            'model_state_dict': self.actor_critic.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metadata': _to_python_types(metadata or {}),
        }
        torch.save(payload, checkpoint_path)

    def load_checkpoint(
        self,
        checkpoint_path: Path,
        map_location: str = 'cpu',
        load_optimizer_state: bool = True,
        strict_shape_check: bool = True,
    ) -> dict:
        payload = torch.load(checkpoint_path, map_location=map_location, weights_only=False)
        metadata = payload.get('metadata', {})
        if strict_shape_check:
            saved_obs_dim = metadata.get('observation_dimension')
            saved_num_actions = metadata.get('number_of_actions')
            if saved_obs_dim is not None and saved_obs_dim != self.observation_dimension:
                raise ValueError(
                    f'Checkpoint observation dimension {saved_obs_dim} does not match current {self.observation_dimension}. '
                    'Use a different checkpoint lineage.'
                )
            if saved_num_actions is not None and saved_num_actions != self.number_of_actions:
                raise ValueError(
                    f'Checkpoint action count {saved_num_actions} does not match current {self.number_of_actions}. '
                    'Use a different checkpoint lineage.'
                )
        self.actor_critic.load_state_dict(payload['model_state_dict'])
        if load_optimizer_state and 'optimizer_state_dict' in payload:
            self.optimizer.load_state_dict(payload['optimizer_state_dict'])
        return metadata



def _to_python_types(value):
    if isinstance(value, dict):
        return {key: _to_python_types(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_python_types(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    return value


class RandomAgent:
    """Simple random baseline for comparison."""

    def __init__(self, number_of_actions: int) -> None:
        self.number_of_actions = number_of_actions

    def choose_evaluation_actions(
        self,
        observation_by_callsign: dict[str, np.ndarray],
    ) -> dict[str, int]:
        return {
            callsign: random.randrange(self.number_of_actions)
            for callsign in observation_by_callsign.keys()
        }


## Curriculum configuration and rollout helpers

The curriculum deliberately keeps these fixed across stages:

- decentralized control
- `extra_minimal` state encoder
- `k_nearest_aircraft = 1`
- lateral-only action set
- same shared policy network shape

It now uses a more progress-oriented reward mix:

- a much lighter centreline shaping term
- `lateral_next_fix_proximity_dist_exp` to reward getting closer to the next fix
- the existing safety penalty
- `expeditious_linear` to reward reducing distance-to-exit
- a stronger custom `route_progress_terminal_reward` that gives:
  - a positive bonus for correct exit
  - a per-step progress signal
  - a timeout penalty for loitering

That is intended to make circling near the entry fix much less attractive.


In [4]:
def make_training_config(
    env_cls,
    num_aircraft: int,
    k_nearest_aircraft: int = 1,
    enable_vertical_actions: bool = False,
    scenario_duration_seconds: int = 1800,
    scenario_cls: str = 'tactical',
    scenario_args: dict | None = None,
    exit_window_width_nmi: float = 5.0,
    reward_coeff_overrides: list[float] | None = None,
) -> EnvConfig:
    config = env_cls.get_default_env_config(ViewType.DECENTRALIZED)
    config.airspace_config['exit_window_width'] = exit_window_width_nmi

    config.state_repr_config = {
        'encoder_cls': 'extra_minimal',
        'k_nearest_aircraft': k_nearest_aircraft,
    }

    config.action_config = {
        'simple_heading_left': True,
        'simple_heading_right': True,
        'simple_heading_route_parallel': True,
        'simple_fl_climb': enable_vertical_actions,
        'simple_fl_descent': enable_vertical_actions,
        'simple_fl_exit': False,
    }

    reward_fns = [
        'position_status_const',
        'lateral_centreline_distance_shaped',
        'lateral_next_fix_proximity_dist_exp',
        'safety_simple_avoidance_exp',
        'expeditious_linear',
        'route_progress_terminal_reward',
        'anti_loiter_route_rejoin_reward',
        'route_parallel_exp',
        'action_penalty_thresh',
    ]
    reward_coeffs = [1.0, 0.06, 0.9, 1.6, 2.2, 3.5, 1.6, 0.8, 0.02]
    if reward_coeff_overrides is not None:
        reward_coeffs = list(reward_coeff_overrides)

    if num_aircraft <= 1:
        reward_fns.append('lateral_termination_check_sac_env')
    else:
        reward_fns.append('lateral_termination_check_mac_env')

    if len(reward_coeffs) == len(reward_fns) - 1:
        reward_coeffs.append(0.05)
    elif len(reward_coeffs) != len(reward_fns):
        raise ValueError(
            'reward_coeff_overrides must match either the base reward count '
            'or the full reward count including termination shaping.'
        )

    config.reward_config = {
        'fns': reward_fns,
        'coeffs': reward_coeffs,
    }

    config.view_config = {
        'type': ViewType.DECENTRALIZED.value,
        'decentralized_params': {},
    }

    if scenario_args is None:
        scenario_args = {
            'num_aircraft': num_aircraft,
            'balance': [0.0, 0.0, 1.0],
        }

    config.scenario_config = {
        'cls': scenario_cls,
        'args': scenario_args,
    }

    config.scenario_duration = scenario_duration_seconds
    return config


def compute_returns_and_advantages(
    rewards: list[float],
    values: list[torch.Tensor],
    dones: list[bool],
    discount_factor_gamma: float,
    gae_lambda: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
    values_tensor = torch.stack(values).detach().float()
    dones_tensor = torch.tensor(dones, dtype=torch.float32)

    advantages = torch.zeros_like(rewards_tensor)
    last_gae = torch.tensor(0.0)

    for timestep in reversed(range(len(rewards))):
        if timestep == len(rewards) - 1:
            next_value = torch.tensor(0.0)
        else:
            next_value = values_tensor[timestep + 1]

        next_nonterminal = 1.0 - dones_tensor[timestep]
        delta = (
            rewards_tensor[timestep]
            + discount_factor_gamma * next_value * next_nonterminal
            - values_tensor[timestep]
        )
        last_gae = delta + discount_factor_gamma * gae_lambda * next_nonterminal * last_gae
        advantages[timestep] = last_gae

    returns = advantages + values_tensor
    return returns, advantages


def run_one_training_episode(
    environment,
    agent: PPOAgent,
    random_seed: int,
    discount_factor_gamma: float,
    gae_lambda: float,
) -> tuple[float, int, dict[str, float] | None]:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = environment.reset(seed=random_seed)

    episode_is_done = False
    episode_step_count = 0
    episode_total_reward = 0.0

    observations: list[torch.Tensor] = []
    actions: list[torch.Tensor] = []
    old_log_probabilities: list[torch.Tensor] = []
    values: list[torch.Tensor] = []
    rewards: list[float] = []
    dones: list[bool] = []

    while not episode_is_done:
        action_by_callsign: dict[str, int] = {}
        step_rollout_rows: list[tuple[str, torch.Tensor, int, torch.Tensor, torch.Tensor]] = []

        for callsign, observation_vector in observation_by_callsign.items():
            action_int, observation_tensor, action_log_probability, state_value = agent.choose_training_action(observation_vector)
            action_by_callsign[callsign] = action_int
            step_rollout_rows.append(
                (
                    callsign,
                    observation_tensor,
                    action_int,
                    action_log_probability.detach(),
                    state_value.detach(),
                )
            )

        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = environment.step(action_by_callsign)

        timestep_reward = float(sum(reward_by_callsign.values())) if reward_by_callsign else 0.0
        timestep_done = all(done_by_callsign.values()) if done_by_callsign else True

        for callsign, observation_tensor, action_int, action_log_probability, state_value in step_rollout_rows:
            observations.append(observation_tensor)
            actions.append(torch.tensor(action_int))
            old_log_probabilities.append(action_log_probability)
            values.append(state_value)
            rewards.append(float(reward_by_callsign.get(callsign, 0.0)))
            dones.append(
                bool(done_by_callsign.get(callsign, False))
                or bool(truncated_by_callsign.get(callsign, False))
            )

        _ = truncated_by_callsign
        episode_total_reward += timestep_reward
        episode_is_done = timestep_done
        observation_by_callsign = next_observation_by_callsign
        episode_step_count += 1

    returns, advantages = compute_returns_and_advantages(
        rewards=rewards,
        values=values,
        dones=dones,
        discount_factor_gamma=discount_factor_gamma,
        gae_lambda=gae_lambda,
    )

    update_metrics = agent.update_from_trajectory(
        observations=observations,
        actions=actions,
        old_log_probabilities=old_log_probabilities,
        returns=returns,
        advantages=advantages,
    )

    return episode_total_reward, episode_step_count, update_metrics


def run_one_evaluation_episode(environment, evaluation_agent, random_seed: int) -> tuple[float, int]:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = environment.reset(seed=random_seed)

    episode_is_done = False
    episode_step_count = 0
    episode_total_reward = 0.0

    while not episode_is_done:
        action_by_callsign = evaluation_agent.choose_evaluation_actions(observation_by_callsign)
        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = environment.step(action_by_callsign)

        _ = truncated_by_callsign
        episode_total_reward += float(sum(reward_by_callsign.values())) if reward_by_callsign else 0.0
        episode_is_done = all(done_by_callsign.values()) if done_by_callsign else True
        observation_by_callsign = next_observation_by_callsign
        episode_step_count += 1

    return episode_total_reward, episode_step_count


def evaluate_agent_over_seeds(environment, evaluation_agent, evaluation_seeds: list[int]) -> dict:
    rewards: list[float] = []
    steps: list[int] = []

    for random_seed in evaluation_seeds:
        total_reward, step_count = run_one_evaluation_episode(environment, evaluation_agent, random_seed)
        rewards.append(total_reward)
        steps.append(step_count)

    return {
        'seeds': evaluation_seeds,
        'rewards': rewards,
        'steps': steps,
        'mean_reward': float(np.mean(rewards)),
        'std_reward': float(np.std(rewards)),
        'mean_steps': float(np.mean(steps)),
    }


def render_evaluation_rollout_to_gif(
    env_cls,
    stage_num_aircraft: int,
    enable_vertical_actions: bool,
    scenario_duration_seconds: int,
    agent: PPOAgent,
    random_seed: int,
    render_dir: Path,
    gif_name: str,
    render_every_n_steps: int = 5,
    gif_frame_duration_seconds: float = 0.2,
    scenario_cls: str = 'tactical',
    scenario_args: dict | None = None,
    exit_window_width_nmi: float = 5.0,
    reward_coeff_overrides: list[float] | None = None,
) -> dict[str, Path]:
    render_config = make_training_config(
        env_cls=env_cls,
        num_aircraft=stage_num_aircraft,
        enable_vertical_actions=enable_vertical_actions,
        scenario_duration_seconds=scenario_duration_seconds,
        scenario_cls=scenario_cls,
        scenario_args=scenario_args,
        exit_window_width_nmi=exit_window_width_nmi,
        reward_coeff_overrides=reward_coeff_overrides,
    )
    render_config.radar_config['display_actions'] = True
    render_config.radar_config['render_dir'] = str(render_dir)
    render_config.radar_config['prefix'] = 'frame'

    if render_dir.exists():
        shutil.rmtree(render_dir)
    render_dir.mkdir(parents=True, exist_ok=True)

    render_environment = env_cls(config=render_config)
    render_environment.set_render_mode('file')

    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = render_environment.reset(seed=random_seed)
    action_formatter_map = render_environment.get_action_parser().action_formatter_map
    action_trace: list[dict] = []
    render_environment.render()

    episode_is_done = False
    step_index = 0

    while not episode_is_done:
        action_by_callsign = agent.choose_evaluation_actions(observation_by_callsign)
        action_trace.append({
            'step': step_index,
            'actions': {
                callsign: action_formatter_map.get(action_int, str(action_int))
                for callsign, action_int in action_by_callsign.items()
            },
        })
        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = render_environment.step(action_by_callsign)
        _ = reward_by_callsign, truncated_by_callsign
        step_index += 1
        episode_is_done = all(done_by_callsign.values()) if done_by_callsign else True
        if step_index % render_every_n_steps == 0 or episode_is_done:
            render_environment.render()
        observation_by_callsign = next_observation_by_callsign

    png_frames = sorted(render_dir.glob(f"{render_config.radar_config['prefix']}_*.png"))
    if not png_frames:
        raise RuntimeError(
            f'No rendered PNG frames were written to {render_dir}. ' 
            'Expected at least one frame before GIF generation.'
        )

    gif_path = render_dir / f'{gif_name}.gif'
    frames = [imageio.v3.imread(frame_path) for frame_path in png_frames]
    imageio.mimsave(gif_path, frames, duration=gif_frame_duration_seconds, loop=0)

    action_trace_path = render_dir / f'{gif_name}_actions.json'
    action_trace_path.write_text(json.dumps(action_trace, indent=2))

    render_environment.close()
    return {
        'gif_path': gif_path,
        'action_trace_path': action_trace_path,
    }


## Curriculum stages and CPU-friendly defaults

These defaults are intended for a standard desktop CPU:

- `hidden_units = 128`
- `ppo_epochs = 3`
- modest per-stage episode counts
- evaluation on 10 held-out seeds
- shorter early-stage horizons so loitering is more expensive

The stage ordering reflects the recommendation from the earlier discussion:

1. `SectorIEnv`, 1 aircraft
2. `SectorIEnv`, 2 aircraft
3. `SectorIEnv`, 3 aircraft
4. `SectorXPlusEnv`, 2 aircraft


In [5]:
global_seed = 7
learning_rate = 3e-4
hidden_units = 128
discount_factor_gamma = 0.99
gae_lambda = 0.95
clip_epsilon = 0.2
value_loss_coefficient = 0.5
entropy_coefficient = 0.01
ppo_epochs = 3
periodic_eval_interval = 10
heldout_evaluation_seeds = list(range(300, 310))
render_every_n_steps = 5
gif_frame_duration_seconds = 0.2

sector_i_exit_window_width_nmi = 5.0
sector_i_forward_route = ['FIRE', 'EARTH', 'WATER', 'AIR', 'SPIRIT']
sector_i_reverse_route = list(reversed(sector_i_forward_route))

curriculum_stages = [
    {
        'stage_name': 'sector_i_1ac_lateral',
        'env_cls': SectorIEnv,
        'num_aircraft': 1,
        'enable_vertical_actions': False,
        'scenario_duration_seconds': 900,
        'training_episodes': 60,
        'training_seed_start': 100,
        'scenario_cls': 'tactical',
        'reward_coeff_overrides': [1.0, 0.06, 0.9, 1.6, 2.2, 3.5, 1.6, 0.8, 0.02, 0.05],
        'scenario_args': {
            'num_aircraft': 1,
            'balance': [0.0, 0.0, 1.0],
        },
        'exit_window_width_nmi': sector_i_exit_window_width_nmi,
    },
    {
        'stage_name': 'sector_i_2ac_head_on_wide_gap_lateral',
        'env_cls': SectorIEnv,
        'num_aircraft': 2,
        'enable_vertical_actions': False,
        'scenario_duration_seconds': 1500,
        'training_episodes': 140,
        'training_seed_start': 2000,
        'scenario_cls': 'fixed_sequence',
        'reward_coeff_overrides': [1.0, 0.04, 0.75, 4.2, 1.4, 2.4, 1.8, 1.0, 0.02, 0.05],
        'scenario_args': {
            'aircraft_specs': [
                {
                    'callsign': 'AIR0',
                    'route_filed': sector_i_forward_route,
                    'start_time_seconds': 0,
                    'speed_tas': 380.0,
                    'entry_fl': 200.0,
                    'exit_fl': 200.0,
                },
                {
                    'callsign': 'AIR1',
                    'route_filed': sector_i_reverse_route,
                    'start_time_seconds': 150,
                    'speed_tas': 380.0,
                    'entry_fl': 200.0,
                    'exit_fl': 200.0,
                },
            ],
        },
        'exit_window_width_nmi': sector_i_exit_window_width_nmi,
    },
    {
        'stage_name': 'sector_i_2ac_head_on_bridge_gap_lateral',
        'env_cls': SectorIEnv,
        'num_aircraft': 2,
        'enable_vertical_actions': False,
        'scenario_duration_seconds': 1500,
        'training_episodes': 140,
        'training_seed_start': 3000,
        'scenario_cls': 'fixed_sequence',
        'reward_coeff_overrides': [1.0, 0.04, 0.75, 4.4, 1.35, 2.3, 1.9, 1.05, 0.02, 0.05],
        'scenario_args': {
            'aircraft_specs': [
                {
                    'callsign': 'AIR0',
                    'route_filed': sector_i_forward_route,
                    'start_time_seconds': 0,
                    'speed_tas': 380.0,
                    'entry_fl': 200.0,
                    'exit_fl': 200.0,
                },
                {
                    'callsign': 'AIR1',
                    'route_filed': sector_i_reverse_route,
                    'start_time_seconds': 120,
                    'speed_tas': 380.0,
                    'entry_fl': 200.0,
                    'exit_fl': 200.0,
                },
            ],
        },
        'exit_window_width_nmi': sector_i_exit_window_width_nmi,
    },
    {
        'stage_name': 'sector_i_2ac_head_on_medium_gap_lateral',
        'env_cls': SectorIEnv,
        'num_aircraft': 2,
        'enable_vertical_actions': False,
        'scenario_duration_seconds': 1550,
        'training_episodes': 150,
        'training_seed_start': 4000,
        'scenario_cls': 'fixed_sequence',
        'reward_coeff_overrides': [1.0, 0.04, 0.75, 5.0, 1.2, 2.1, 2.0, 1.15, 0.02, 0.05],
        'scenario_args': {
            'aircraft_specs': [
                {
                    'callsign': 'AIR0',
                    'route_filed': sector_i_forward_route,
                    'start_time_seconds': 0,
                    'speed_tas': 380.0,
                    'entry_fl': 200.0,
                    'exit_fl': 200.0,
                },
                {
                    'callsign': 'AIR1',
                    'route_filed': sector_i_reverse_route,
                    'start_time_seconds': 90,
                    'speed_tas': 380.0,
                    'entry_fl': 200.0,
                    'exit_fl': 200.0,
                },
            ],
        },
        'exit_window_width_nmi': sector_i_exit_window_width_nmi,
    },
]

# Same-direction bridge stages are intentionally disabled in this notebook.
# The current focus is to stabilize pure head-on avoidance and route rejoin.
# 3-aircraft Sector I and SectorXPlus remain disabled until that behavior is stable.

curriculum_checkpoint_dir = Path.cwd() / 'checkpoints' / 'ppo_curriculum_agent'
resume_from_checkpoint: Path | None = None
resume_from_stage_index = 0
load_optimizer_state = True
history_json_path = curriculum_checkpoint_dir / 'curriculum_history.json'


## Initialize the first stage and optionally resume a checkpoint

The checkpoint loader will reject incompatible observation/action shapes.
That is deliberate: if you change the action set, you should start a new checkpoint lineage.


In [6]:
random.seed(global_seed)
np.random.seed(global_seed)
torch.manual_seed(global_seed)

first_stage = curriculum_stages[resume_from_stage_index]
first_config = make_training_config(
    env_cls=first_stage['env_cls'],
    num_aircraft=first_stage['num_aircraft'],
    enable_vertical_actions=first_stage['enable_vertical_actions'],
    scenario_duration_seconds=first_stage['scenario_duration_seconds'],
    scenario_cls=first_stage['scenario_cls'],
    scenario_args=first_stage['scenario_args'],
    exit_window_width_nmi=first_stage['exit_window_width_nmi'],
    reward_coeff_overrides=first_stage.get('reward_coeff_overrides'),
)
first_environment = first_stage['env_cls'](config=first_config)
observation_dimension = first_environment.observation_space.shape[0]
number_of_actions = first_environment.action_space.n
first_environment.close()

agent = PPOAgent(
    observation_dimension=observation_dimension,
    number_of_actions=number_of_actions,
    learning_rate=learning_rate,
    hidden_units=hidden_units,
    clip_epsilon=clip_epsilon,
    value_loss_coefficient=value_loss_coefficient,
    entropy_coefficient=entropy_coefficient,
    ppo_epochs=ppo_epochs,
)
random_agent = RandomAgent(number_of_actions=number_of_actions)

if resume_from_checkpoint is not None:
    loaded_metadata = agent.load_checkpoint(
        resume_from_checkpoint,
        load_optimizer_state=load_optimizer_state,
        strict_shape_check=True,
    )
    print('Resumed from checkpoint:', resume_from_checkpoint)
    print('Loaded metadata:', loaded_metadata)

print('Global seed:', global_seed)
print('Shared observation dimension:', observation_dimension)
print('Shared action count:', number_of_actions)


Global seed: 7
Shared observation dimension: 4
Shared action count: 4


## Run the curriculum

Each stage:

- builds a fresh environment for that stage
- keeps the same PPO weights in memory
- saves `latest.pt` and `best.pt` inside a stage-specific checkpoint directory
- writes stage history to `curriculum_history.json`
- warms the next stage from the best checkpoint of the current stage


In [7]:
curriculum_history: list[dict] = []

for stage_index in range(resume_from_stage_index, len(curriculum_stages)):
    stage = curriculum_stages[stage_index]
    stage_name = stage['stage_name']
    stage_dir = curriculum_checkpoint_dir / stage_name
    latest_checkpoint_path = stage_dir / 'latest.pt'
    best_checkpoint_path = stage_dir / 'best.pt'

    config = make_training_config(
        env_cls=stage['env_cls'],
        num_aircraft=stage['num_aircraft'],
        enable_vertical_actions=stage['enable_vertical_actions'],
        scenario_duration_seconds=stage['scenario_duration_seconds'],
        scenario_cls=stage['scenario_cls'],
        scenario_args=stage['scenario_args'],
        exit_window_width_nmi=stage['exit_window_width_nmi'],
        reward_coeff_overrides=stage.get('reward_coeff_overrides'),
    )
    environment = stage['env_cls'](config=config)

    stage_observation_dimension = environment.observation_space.shape[0]
    stage_number_of_actions = environment.action_space.n
    if stage_observation_dimension != agent.observation_dimension:
        raise ValueError(
            f'Stage {stage_name} observation dimension {stage_observation_dimension} does not match current agent {agent.observation_dimension}.'
        )
    if stage_number_of_actions != agent.number_of_actions:
        raise ValueError(
            f'Stage {stage_name} action count {stage_number_of_actions} does not match current agent {agent.number_of_actions}.'
        )

    training_rewards: list[float] = []
    training_steps: list[int] = []
    training_policy_losses: list[float] = []
    training_value_losses: list[float] = []
    training_entropies: list[float] = []
    training_total_losses: list[float] = []
    periodic_eval_episodes: list[int] = []
    periodic_eval_learned_mean_rewards: list[float] = []
    periodic_eval_learned_std_rewards: list[float] = []
    periodic_eval_random_mean_rewards: list[float] = []
    periodic_eval_random_std_rewards: list[float] = []

    best_mean_evaluation_reward = float('-inf')
    best_stage_metadata: dict = {}

    print()
    print('=== Starting stage:', stage_name, '===')

    for episode_index in range(stage['training_episodes']):
        random_seed = stage['training_seed_start'] + episode_index
        total_reward, step_count, update_metrics = run_one_training_episode(
            environment=environment,
            agent=agent,
            random_seed=random_seed,
            discount_factor_gamma=discount_factor_gamma,
            gae_lambda=gae_lambda,
        )

        training_rewards.append(total_reward)
        training_steps.append(step_count)
        training_policy_losses.append(float('nan') if update_metrics is None else update_metrics['policy_loss'])
        training_value_losses.append(float('nan') if update_metrics is None else update_metrics['value_loss'])
        training_entropies.append(float('nan') if update_metrics is None else update_metrics['entropy'])
        training_total_losses.append(float('nan') if update_metrics is None else update_metrics['total_loss'])

        print(
            '[train]',
            f'stage={stage_name}',
            f'episode={episode_index:03d}',
            f'seed={random_seed}',
            f'reward={total_reward:.3f}',
            f'steps={step_count}',
        )

        should_run_periodic_eval = (
            (episode_index + 1) % periodic_eval_interval == 0
            or episode_index == stage['training_episodes'] - 1
        )

        if should_run_periodic_eval:
            learned_eval = evaluate_agent_over_seeds(
                environment=environment,
                evaluation_agent=agent,
                evaluation_seeds=heldout_evaluation_seeds,
            )
            random_eval = evaluate_agent_over_seeds(
                environment=environment,
                evaluation_agent=random_agent,
                evaluation_seeds=heldout_evaluation_seeds,
            )

            periodic_eval_episodes.append(episode_index + 1)
            periodic_eval_learned_mean_rewards.append(learned_eval['mean_reward'])
            periodic_eval_learned_std_rewards.append(learned_eval['std_reward'])
            periodic_eval_random_mean_rewards.append(random_eval['mean_reward'])
            periodic_eval_random_std_rewards.append(random_eval['std_reward'])

            metadata = {
                'stage_name': stage_name,
                'stage_index': stage_index,
                'environment': stage['env_cls'].__name__,
                'view_type': 'decentralized',
                'num_aircraft': stage['num_aircraft'],
                'scenario_cls': stage['scenario_cls'],
                'scenario_args': stage['scenario_args'],
                'exit_window_width_nmi': stage['exit_window_width_nmi'],
                'enable_vertical_actions': stage['enable_vertical_actions'],
                'reward_coeff_overrides': stage.get('reward_coeff_overrides'),
                'observation_dimension': stage_observation_dimension,
                'number_of_actions': stage_number_of_actions,
                'episode': episode_index + 1,
                'learned_mean_reward': learned_eval['mean_reward'],
                'learned_std_reward': learned_eval['std_reward'],
                'random_mean_reward': random_eval['mean_reward'],
                'random_std_reward': random_eval['std_reward'],
                'evaluation_seeds': heldout_evaluation_seeds,
            }
            agent.save_checkpoint(latest_checkpoint_path, metadata=metadata)

            if learned_eval['mean_reward'] > best_mean_evaluation_reward:
                best_mean_evaluation_reward = learned_eval['mean_reward']
                best_stage_metadata = metadata
                agent.save_checkpoint(best_checkpoint_path, metadata=metadata)
                checkpoint_note = 'new best checkpoint'
            else:
                checkpoint_note = 'latest checkpoint only'

            print(
                '[periodic-eval]',
                f'stage={stage_name}',
                f'episode={episode_index + 1:03d}',
                f'learned_mean_reward={learned_eval["mean_reward"]:.3f}',
                f'random_mean_reward={random_eval["mean_reward"]:.3f}',
                checkpoint_note,
            )

    stage_history = {
        'stage_name': stage_name,
        'environment': stage['env_cls'].__name__,
        'num_aircraft': stage['num_aircraft'],
        'scenario_cls': stage['scenario_cls'],
        'scenario_args': stage['scenario_args'],
        'exit_window_width_nmi': stage['exit_window_width_nmi'],
        'reward_coeff_overrides': stage.get('reward_coeff_overrides'),
        'best_mean_evaluation_reward': best_mean_evaluation_reward,
        'best_checkpoint_path': str(best_checkpoint_path),
        'latest_checkpoint_path': str(latest_checkpoint_path),
        'training_rewards': training_rewards,
        'training_steps': training_steps,
        'training_policy_losses': training_policy_losses,
        'training_value_losses': training_value_losses,
        'training_entropies': training_entropies,
        'training_total_losses': training_total_losses,
        'periodic_eval_episodes': periodic_eval_episodes,
        'periodic_eval_learned_mean_rewards': periodic_eval_learned_mean_rewards,
        'periodic_eval_random_mean_rewards': periodic_eval_random_mean_rewards,
        'best_stage_metadata': best_stage_metadata,
    }
    curriculum_history.append(stage_history)

    curriculum_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    with history_json_path.open('w', encoding='utf-8') as fp:
        json.dump(
            curriculum_history,
            fp,
            indent=2,
            default=lambda obj: obj.item() if isinstance(obj, np.generic) else str(obj),
        )

    if best_checkpoint_path.exists():
        reloaded_metadata = agent.load_checkpoint(
            best_checkpoint_path,
            load_optimizer_state=False,
            strict_shape_check=True,
        )
        print('Loaded best checkpoint for next stage warm start:', reloaded_metadata)

    environment.close()



=== Starting stage: sector_i_1ac_lateral ===


ValueError: zip() argument 2 is longer than argument 1

## Stage summary plots

This plot keeps the same “quick-look” style as the older PPO notebooks,
but compares progress stage by stage.


In [ ]:
if not curriculum_history:
    raise ValueError('No curriculum history recorded yet. Run the curriculum loop first.')

num_stages = len(curriculum_history)
fig, axes = plt.subplots(num_stages, 2, figsize=(15, 5 * num_stages))
if num_stages == 1:
    axes = np.asarray([axes])

for row_index, stage_history in enumerate(curriculum_history):
    training_rewards = stage_history['training_rewards']
    plot_window = min(10, len(training_rewards))
    if len(training_rewards) >= plot_window and plot_window > 0:
        smoothed_rewards = np.convolve(
            np.asarray(training_rewards, dtype=float),
            np.ones(plot_window) / plot_window,
            mode='valid',
        )
    else:
        smoothed_rewards = np.array([])

    training_episode_indices = np.arange(1, len(training_rewards) + 1)
    eval_episodes = np.asarray(stage_history['periodic_eval_episodes'])
    learned_mean = np.asarray(stage_history['periodic_eval_learned_mean_rewards'])
    random_mean = np.asarray(stage_history['periodic_eval_random_mean_rewards'])

    ax_left = axes[row_index, 0]
    ax_left.plot(training_episode_indices, training_rewards, marker='o', alpha=0.25, label='raw reward')
    if len(smoothed_rewards) > 0:
        ax_left.plot(
            np.arange(plot_window, len(training_rewards) + 1),
            smoothed_rewards,
            linewidth=2.5,
            label=f'moving average (window={plot_window})',
        )
    ax_left.set_title(f"{stage_history['stage_name']} training reward")
    ax_left.set_xlabel('Episode')
    ax_left.set_ylabel('Total reward')
    ax_left.grid(alpha=0.3)
    ax_left.legend()

    ax_right = axes[row_index, 1]
    ax_right.plot(eval_episodes, learned_mean, marker='o', label='learned policy')
    ax_right.plot(eval_episodes, random_mean, marker='s', label='random baseline')
    ax_right.set_title(f"{stage_history['stage_name']} periodic evaluation")
    ax_right.set_xlabel('Training episode')
    ax_right.set_ylabel('Mean evaluation reward')
    ax_right.grid(alpha=0.3)
    ax_right.legend()

fig.suptitle('PPO Curriculum Training Summary', fontsize=16)
fig.tight_layout()
plt.show()


## Render best-checkpoint GIFs for each curriculum stage

This renders one evaluation GIF for the best checkpoint from each stage.
The final stage is included automatically.


In [ ]:
if not curriculum_history:
    raise ValueError('No curriculum history recorded yet. Run the curriculum loop first.')

stage_render_artifacts: dict[str, dict[str, Path]] = {}
gif_seed = heldout_evaluation_seeds[0]

for stage_history in curriculum_history:
    stage_name_to_render = stage_history['stage_name']
    stage_config = next(stage for stage in curriculum_stages if stage['stage_name'] == stage_name_to_render)
    best_checkpoint_path = Path(stage_history['best_checkpoint_path'])

    if not best_checkpoint_path.exists():
        print(f'Skipping {stage_name_to_render}: checkpoint not found at {best_checkpoint_path}')
        continue

    loaded_metadata = agent.load_checkpoint(
        best_checkpoint_path,
        load_optimizer_state=False,
        strict_shape_check=True,
    )
    print()
    print(f'Rendering stage {stage_name_to_render} checkpoint metadata: {loaded_metadata}')

    render_dir = Path.cwd() / 'renders' / f'{stage_name_to_render}_eval'
    render_artifacts = render_evaluation_rollout_to_gif(
        env_cls=stage_config['env_cls'],
        stage_num_aircraft=stage_config['num_aircraft'],
        enable_vertical_actions=stage_config['enable_vertical_actions'],
        scenario_duration_seconds=stage_config['scenario_duration_seconds'],
        agent=agent,
        random_seed=gif_seed,
        render_dir=render_dir,
        gif_name=f'{stage_name_to_render}_best_eval_seed_{gif_seed}',
        render_every_n_steps=render_every_n_steps,
        gif_frame_duration_seconds=gif_frame_duration_seconds,
        scenario_cls=stage_config['scenario_cls'],
        scenario_args=stage_config['scenario_args'],
        exit_window_width_nmi=stage_config['exit_window_width_nmi'],
        reward_coeff_overrides=stage_config.get('reward_coeff_overrides'),
    )

    stage_render_artifacts[stage_name_to_render] = render_artifacts
    print(f"Saved GIF to: {render_artifacts['gif_path']}")
    print(f"Saved action trace to: {render_artifacts['action_trace_path']}")
    display(Image(filename=str(render_artifacts['gif_path'])))

stage_render_artifacts


## Practical notes

This notebook is the recommended place to try:

- more episodes per stage
- different stage orderings
- different aircraft counts
- checkpoint resume across notebook sessions

Do not add climb/descent into this notebook’s checkpoint lineage unless you are willing
to start a new branch of experiments.
